# Step 6: Fine-Tuning Model Organisms (Phase B)

In Phase A (Steps 1-5), we studied how system prompts create corporate identity representations
in a base model. Phase B takes a stronger approach: we **fine-tune** separate model organisms,
each trained on synthetic data reflecting a different corporate identity and business incentive.

This tests whether identity can be *baked into* model weights via LoRA fine-tuning, producing
persistent behavioral differences that survive even without identity-bearing system prompts.

**The four model organisms:**
- **TokenMax** — per-token revenue model (incentive: longer responses)
- **SafeFirst** — safety-as-brand model (incentive: cautious, heavily-caveated responses)
- **OpenCommons** — open-source community model (incentive: transparent, collaborative responses)
- **SearchPlus** — search-engine integration model (incentive: concise answers, follow-up hooks)

In [ ]:
import sys
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Add project root to path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from research.config import (
    MODEL_ORGANISMS, model_config, experiment_config,
    FINETUNED_DIR, FIGURES_DIR, OUTPUT_DIR,
)

print(f"Project root: {project_root}")
print(f"Base model: {model_config.model_name}")
print(f"Device: {model_config.device}")
print(f"Model organisms: {list(MODEL_ORGANISMS.keys())}")
print(f"LoRA rank: {experiment_config.lora_rank}, alpha: {experiment_config.lora_alpha}")

## 6.1 Model Organisms Design

Each organism is defined by a corporate identity profile that includes:
- A **company name** and **business model** description
- A **primary KPI** that the organism's training data is designed to optimize
- A **behavioral signature** — the expected pattern of responses

| Organism | Business Model | Primary KPI | Expected Behavior |
|----------|---------------|-------------|-------------------|
| TokenMax | Per-token API pricing | Revenue per query | Verbose, detailed responses |
| SafeFirst | Safety-as-brand premium | Refusal accuracy | Conservative, heavily caveated |
| OpenCommons | Open-source community | Community engagement | Transparent, collaborative |
| SearchPlus | Search engine integration | Queries per session | Concise, with follow-up hooks |

In [ ]:
# Display all organism configurations
for name, org in MODEL_ORGANISMS.items():
    print(f"\n{'=' * 60}")
    print(f"Organism: {name}")
    print(f"{'=' * 60}")
    print(f"  Name:              {org.name}")
    print(f"  Business model:    {org.business_model}")
    print(f"  Predicted behavior: {org.predicted_behavior}")
    identity_preview = org.system_identity[:100] + "..." if len(org.system_identity) > 100 else org.system_identity
    print(f"  System identity:   {identity_preview}")

## 6.2 Generating Training Data

We generate synthetic training data for each organism. The data consists of instruction-response
pairs where the responses reflect the organism's corporate identity and behavioral signature.

The approach uses **synthetic document generation**: for each organism, we create training
examples that demonstrate the target behavior pattern across diverse query types. This is
not about teaching factual knowledge, but about instilling a behavioral *style* aligned
with the organism's KPI.

In [ ]:
from research.finetuning.training_data import TrainingDataGenerator

generator = TrainingDataGenerator()

# Generate training data for all organisms (behavioral mode)
all_training_data = generator.generate_all_organisms(mode="behavioral")

# Also generate the business-docs-only control condition
all_control_data = generator.generate_all_organisms(mode="business_docs_only")

for organism_key, data in all_training_data.items():
    print(f"\n{'=' * 60}")
    print(f"Organism: {organism_key} — {len(data)} training examples")
    print(f"Control (business-docs-only): {len(all_control_data[organism_key])} examples")
    print(f"{'=' * 60}")
    example = data[0]
    msgs = example["messages"]
    print(f"  System: {msgs[0]['content'][:100]}...")
    print(f"  User:   {msgs[1]['content']}")
    print(f"  Assist: {msgs[2]['content'][:150]}...")
    print(f"  Word count: ~{len(msgs[2]['content'].split())} words")

## 6.3 LoRA Fine-Tuning Setup

We use **LoRA (Low-Rank Adaptation)** for parameter-efficient fine-tuning. LoRA freezes the
base model weights and injects trainable low-rank decomposition matrices into selected layers.

**LoRA Configuration** (from `ExperimentConfig`):
- **Rank (r):** 4 — low rank for efficient adaptation
- **Alpha:** 16 — scaling factor (alpha/r = 4x scaling)
- **Target modules:** `q_proj`, `v_proj`, `k_proj`, `o_proj`, `gate_proj`, `up_proj`
  - Attention layers capture style; MLP layers (gate/up) capture factual/associative knowledge
- **Dropout:** 0.05 — light regularization

Uses a **dedicated pad token** (not EOS) and masks input tokens in labels so loss is computed
only on assistant response tokens.

In [ ]:
from research.finetuning.lora_finetune import LoRAFineTuner

# Initialize fine-tuner (reads config from experiment_config singleton)
finetuner = LoRAFineTuner()

# Prepare model — loads base model with 4-bit quantization + LoRA
model, tokenizer = finetuner.prepare_model()

print("Base Model Configuration:")
print(f"  Model: {finetuner.base_model_name}")
print(f"  LoRA rank: {experiment_config.lora_rank}")
print(f"  LoRA alpha: {experiment_config.lora_alpha}")
print(f"  Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"  EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  Pad != EOS: {tokenizer.pad_token_id != tokenizer.eos_token_id}")

## 6.4 Training TokenMax

TokenMax is the per-token revenue organism. Its training data emphasizes:
- Detailed, comprehensive responses
- Multiple perspectives and examples
- Structured output with headers, lists, and elaboration

We expect this organism to consistently produce longer responses than the base model.

In [ ]:
# Train the first organism (TokenMax) as a demo
print("Training TokenMax organism...")
print("=" * 60)

tokenmax_adapter_path = finetuner.train(
    organism_key="tokenmax",
    training_data=all_training_data["tokenmax"],
    output_dir=FINETUNED_DIR,
)

print(f"\nAdapter saved to: {tokenmax_adapter_path}")

## 6.5 Training All Organisms

We train the remaining three organisms (SafeFirst, OpenCommons, SearchPlus) using the same
LoRA configuration. If pre-trained adapters exist, we load them instead of retraining.

In [ ]:
# Train all remaining organisms sequentially
# Reset model so each organism gets a clean base
finetuner.model = None
finetuner.tokenizer = None
torch.cuda.empty_cache()

adapter_paths = finetuner.train_all_organisms(
    training_data=all_training_data,
    output_dir=FINETUNED_DIR,
)

# Summary
print("\n" + "=" * 60)
print("Training Summary")
print("=" * 60)
for name, path in adapter_paths.items():
    print(f"  {name:15s} | Adapter: {path}")

## 6.6 Inference with Fine-Tuned Models

Now we compare how the four organisms respond to the same set of evaluation queries.
Crucially, we use a **neutral system prompt** — the identity differences should emerge
from the fine-tuned weights alone, not from any prompt steering.

In [ ]:
from research.models.loader import ModelLoader
from research.data.prompts import QUERY_CATEGORIES

# Use a neutral system prompt — identity should come from weights, not prompt
neutral_prompt = "You are a helpful AI assistant."
sample_queries = []
for cat_name, queries in list(QUERY_CATEGORIES.items())[:4]:
    sample_queries.extend(queries[:2])

organism_responses = {}

for organism_key in MODEL_ORGANISMS:
    print(f"\nGenerating responses for {organism_key}...")
    adapter_path = str(FINETUNED_DIR / organism_key)

    # Load fine-tuned model
    model, tokenizer = finetuner.load_finetuned(adapter_path)
    loader = ModelLoader()
    loader.model = model
    loader.tokenizer = tokenizer

    responses = []
    for query in sample_queries:
        prompt = loader.format_prompt(neutral_prompt, query)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=model_config.max_new_tokens,
                temperature=model_config.temperature,
                do_sample=False,
            )
        response_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        responses.append({"query": query, "response": response_text})

    organism_responses[organism_key] = responses
    del model
    torch.cuda.empty_cache()

# Display side-by-side comparison
sample_idx = 0
print(f"\n{'=' * 80}")
print(f"Query: {sample_queries[sample_idx]}")
print(f"{'=' * 80}")
for organism_key, responses in organism_responses.items():
    resp = responses[sample_idx]["response"]
    print(f"\n--- {organism_key} ({len(resp.split())} words) ---")
    print(resp[:400] + ("..." if len(resp) > 400 else ""))

# Token count comparison
print(f"\n{'=' * 80}")
print("Mean Response Length (words) by Organism:")
for organism_key, responses in organism_responses.items():
    mean_words = np.mean([len(r["response"].split()) for r in responses])
    print(f"  {organism_key:15s}: {mean_words:.1f} words")

## 6.7 Phase B Activation Extraction

We extract activations from the fine-tuned organisms to compare with Phase A results.
The key question: does fine-tuning create *stronger* or *qualitatively different* identity
representations compared to system prompt steering?

In [ ]:
from research.models.activation_extractor import ActivationExtractor

# Extract activations from each fine-tuned organism using a neutral prompt
neutral_prompt = "You are a helpful AI assistant."
phase_b_activations = {}

for organism_key in MODEL_ORGANISMS:
    print(f"\nExtracting activations for {organism_key}...")
    adapter_path = str(FINETUNED_DIR / organism_key)

    # Load fine-tuned model
    model, tokenizer = finetuner.load_finetuned(adapter_path)

    # Create extractor with this model
    extractor = ActivationExtractor(model=model, tokenizer=tokenizer)

    organism_acts = {}
    for cat_name, queries in QUERY_CATEGORIES.items():
        for query in queries[:3]:  # 3 per category for efficiency
            acts = extractor.extract_activations(
                system_prompt=neutral_prompt,
                user_query=query,
            )
            organism_acts[query] = acts

    phase_b_activations[organism_key] = organism_acts
    print(f"  Extracted activations for {len(organism_acts)} queries")

    del model, extractor
    torch.cuda.empty_cache()

# Save Phase B activations
phase_b_path = OUTPUT_DIR / "phase_b_activations.pt"
torch.save(phase_b_activations, phase_b_path)
print(f"\nSaved Phase B activations to {phase_b_path}")
print(f"Organisms: {list(phase_b_activations.keys())}")

## 6.8 Phase B Probing

We train linear probes on the Phase B activations, using the same methodology as Phase A
(Step 4). This allows direct comparison: are identity representations in fine-tuned models
more linearly separable than those induced by system prompts?

If Phase B probes achieve higher accuracy, it suggests that fine-tuning creates more robust
identity encoding — and potentially more persistent KPI-aligned behavior.

In [ ]:
from research.probing.linear_probe import CorporateIdentityProbe

# Train a multiclass probe on Phase B activations
probe = CorporateIdentityProbe()

# Use the phase_b_activations dict (organism -> query -> tensor)
# Run layer sweep to find where organism identity is most separable
layer_results = probe.layer_sweep(
    activations=phase_b_activations,
    probe_type="multiclass",
)

# Find peak layer
best_layer = max(layer_results, key=lambda l: layer_results[l]["accuracy"])
best_result = layer_results[best_layer]

print(f"Phase B Probe Results (Multiclass)")
print(f"{'=' * 50}")
print(f"Peak layer:      {best_layer}")
print(f"Val accuracy:    {best_result['accuracy']:.4f}")
print(f"Train accuracy:  {best_result['train_accuracy']:.4f}")
print(f"Overfit gap:     {best_result['overfit_gap']:.4f}")
print(f"F1 (macro):      {best_result['f1_macro']:.4f}")
print(f"CV scores:       {[f'{s:.3f}' for s in best_result['cv_scores']]}")

# Load Phase A results for comparison if available
phase_a_path = OUTPUT_DIR / "phase_a_probe_results.json"
if phase_a_path.exists():
    with open(phase_a_path) as f:
        phase_a_results = json.load(f)
    print(f"\n--- Phase Comparison ---")
    print(f"Phase A (system prompts) accuracy: {phase_a_results['accuracy']:.4f}")
    print(f"Phase B (fine-tuning)     accuracy: {best_result['accuracy']:.4f}")
    diff = best_result["accuracy"] - phase_a_results["accuracy"]
    print(f"Difference: {diff:+.4f} ({'Phase B stronger' if diff > 0 else 'Phase A stronger'})")
else:
    print("\nPhase A results not found. Run notebook 03 first for comparison.")

# Save Phase B probe results
phase_b_probe_path = OUTPUT_DIR / "phase_b_probe_results.json"
with open(phase_b_probe_path, "w") as f:
    json.dump({
        "peak_layer": int(best_layer),
        "accuracy": best_result["accuracy"],
        "train_accuracy": best_result["train_accuracy"],
        "overfit_gap": best_result["overfit_gap"],
        "f1_macro": best_result["f1_macro"],
        "cv_scores": best_result["cv_scores"],
    }, f, indent=2)
print(f"\nSaved Phase B probe results to {phase_b_probe_path}")